# Brand & Competitor Intelligence Engine - Phase 2
## Exploratory Data Analysis & Data Quality Report

This notebook demonstrates the exploratory analysis layer of the project. It validates the raw collected datasets (Mamaearth vs Minimalist from Reddit), computes completeness and duplicate quality checks, aggregates brand mention and subreddit statistics, extracts word N-grams & TF-IDF metrics, and analyzes temporal trends.

In [ ]:
import os
import sys
import pandas as pd

# Ensure project root is in python path to resolve src imports
sys.path.append(os.path.abspath(".."))

from src.utils.database_helper import DatabaseHelper
from src.utils.config import config
from src.analytics.data_quality import DataQualityAnalyzer
from src.analytics.exploratory_analysis import ExploratoryAnalyzer

### 1. Data Quality Analysis
We initialize the `DataQualityAnalyzer` using our database path to check missing column percentages, duplicate records, and general size metrics.

In [ ]:
db_helper = DatabaseHelper(config.database_path)
dq = DataQualityAnalyzer(db_helper)
metrics = dq.compute_quality_metrics()

# Print console-style summary
dq.print_console_summary(metrics)

### 2. Brand Analytics & Competitor Overlap
We compute total posts, comments, unique active authors, and direct competitor co-mention profiles where both brands are discussed in the same context.

In [ ]:
preprocessed_posts = "../data/preprocessed_posts.csv"
preprocessed_comments = "../data/preprocessed_comments.csv"
reports_dir = "../reports"

eda = ExploratoryAnalyzer(db_helper, preprocessed_posts, preprocessed_comments)
brand_summary = eda.generate_brand_analytics(reports_dir)
print("Brand Summary Table:")
display(brand_summary)

co_mentions = pd.read_csv(os.path.join(reports_dir, "brand_comentions.csv"))
print(f"Direct brand co-mention posts: {len(co_mentions)}")
if not co_mentions.empty:
    display(co_mentions[["title", "subreddit", "score"]].head(5))

### 3. Subreddit Distribution Analysis
This helps identify the most active communities engaging in discussions about these D2C cosmetics brands.

In [ ]:
sub_posts, sub_summary = eda.generate_subreddit_analytics(reports_dir)
print("Top Subreddits by Combined Engagement:")
display(sub_summary.head(10))

### 4. Text Analytics: N-grams Frequencies
Extract top unigrams, bigrams, and trigrams across all preprocessed post and comment bodies.

In [ ]:
text_stats = eda.generate_text_analytics(reports_dir)

print("Top 10 Bigrams:")
display(text_stats["bigrams"].head(10))

print("\nTop 10 Trigrams:")
display(text_stats["trigrams"].head(10))

### 5. Skincare Competitors TF-IDF Comparison
Compute TF-IDF statistics separately for posts mentioning Mamaearth vs Minimalist to identify core distinct themes.

In [ ]:
tfidf_mama, tfidf_mini = eda.generate_tfidf_analytics(reports_dir)

print("Top distinct terms for Mamaearth:")
display(tfidf_mama.head(10))

print("\nTop distinct terms for Minimalist:")
display(tfidf_mini.head(10))

### 6. Time-Series Trends
Aggregates and prints posting volumes grouped by Month.

In [ ]:
posts_time, comments_time = eda.generate_time_series_analytics(reports_dir)

print("Posts per Month Trend:")
display(posts_time)

print("\nComments per Month Trend:")
display(comments_time)

### 7. Plotly Visualization Generation
Save interactive charts to `reports/figures/` as HTML.

In [ ]:
figures_dir = "../reports/figures"
eda.generate_visualizations(figures_dir, text_stats, posts_time, comments_time)
print(f"Visualizations saved to: {os.path.abspath(figures_dir)}")